In [ ]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer, BitsAndBytesConfig
import soundfile as sf

# Configuration for 8-bit quantization
quantization_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,  # Threshold for outlier detection
    llm_int8_skip_modules=["lm_heads"],  # Skip language modeling heads
)

# Configuration for 4-bit quantization (more aggressive)
quantization_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # NormalFloat4 quantization
    bnb_4bit_use_double_quant=True,  # Nested quantization for better compression
    bnb_4bit_compute_dtype=torch.bfloat16,  # Compute dtype for inference
)

# Load model with quantization
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Choose one configuration
model = ParlerTTSForConditionalGeneration.from_pretrained(
    "parler-tts/parler-tts-large-v1",
    quantization_config=quantization_config_8bit,  # or quantization_config_4bit
    device_map="auto",  # Automatically distribute across devices
    torch_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained("parler-tts/parler-tts-large-v1")

In [ ]:
# Test inference
prompt = "Hey, how are you doing today?"
description = "A female speaker delivers a slightly expressive and animated speech with a moderate speed and pitch."

input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
prompt_input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

# Generate audio
with torch.inference_mode():
    generation = model.generate(
        input_ids=input_ids, 
        prompt_input_ids=prompt_input_ids,
        do_sample=True,
        temperature=1.0
    )

audio_arr = generation.cpu().numpy().squeeze()
sf.write("quantized_parler_tts_out.wav", audio_arr, model.config.sampling_rate)

print(f"Model size reduced. Audio generated successfully!")
